<a href="https://colab.research.google.com/github/duguay-michele/Gender-in-Audio-Datasets/blob/main/Audio_Features.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import numpy as np
import librosa
import scipy.signal as sps

# Define parameters
hop_len = 512
nfft = 2048
min_pitchVal = librosa.note_to_hz('C1') # Minimum pitch value
max_pitchVal = librosa.note_to_hz('C7') # Maximum pitch value

# Vocal presence detection function
def vocal_presence(y, sr, nfft=1024, hop_len=256):
    # Features: Root Mean Square Energy (RMSE) and Zero Crossing Rate (ZCR)
    rmse = librosa.feature.rms(y=y, frame_length=nfft, hop_length=hop_len, center=False)[0]
    zcr  = librosa.feature.zero_crossing_rate(y=y, frame_length=nfft, hop_length=hop_len, center=False)[0]

    # Clip-adaptive thresholds (percentile-based) + sanity clamps for robustness
    # Energy (RMS) thresholds
    ste_on  = max(np.percentile(rmse, 65), 0.02)   # Activation threshold for RMS (don't go below 0.02 on normalized audio)
    ste_off = max(np.percentile(rmse, 45), 0.015)  # Deactivation threshold for RMS

    # ZCR thresholds (lower ZCR typically indicates more voiced sound) – invert logic with hysteresis
    zcr_on  = np.clip(np.percentile(zcr, 35), 0.08, 0.18) # Activation threshold for ZCR
    zcr_off = np.clip(np.percentile(zcr, 65), 0.18, 0.30) # Deactivation threshold for ZCR

    # Hysteresis state machine to determine vocal presence frames
    is_vocal = np.zeros_like(rmse, dtype=bool)
    state = False
    for i, (e, z) in enumerate(zip(rmse, zcr)):
        if not state:
            state = (e >= ste_on) and (z <= zcr_on) # Activate if energy is high and ZCR is low
        else:
            state = not ((e <= ste_off) or (z >= zcr_off)) # Deactivate if energy is low or ZCR is high
        is_vocal[i] = state

    # Debounce flicker: Apply a median filter to smooth out short, transient vocal/non-vocal segments
    # Kernel size 'k' is calculated to be approximately 30-50 ms
    k = int(round(0.04 * sr / hop_len)) | 1  # Ensure odd kernel size, ~40 ms
    is_vocal = sps.medfilt(is_vocal.astype(float), kernel_size=max(k, 3)) > 0.5
    return is_vocal


In [ ]:
import parselmouth
import librosa
import os
import numpy as np
import pandas as pd
import scipy.signal as sps

from google.colab import drive
drive.mount('/content/drive')

# Function to extract fmin, fmax, quartiles, and spectral centroid
def extract_features(audio_path): # Removed redundant arguments as they are now global
    y, sr = librosa.load(audio_path)
    # reduce y to 10 seconds in length
    y = y[:sr*10]

    window_length = nfft / sr
    periods_per_window = window_length * min_pitchVal
    ts = hop_len / sr

    # calculate features
    f0, _, _ = librosa.pyin(y, fmin=min_pitchVal, fmax=max_pitchVal,frame_length=nfft, hop_length=hop_len,center=False)
    spectral_centroid = librosa.feature.spectral_centroid(y=y, sr=sr, n_fft=nfft, hop_length=hop_len,center=False)
    S, phase = librosa.magphase(librosa.stft(y,n_fft=nfft, hop_length=hop_len,center=False))
    rms = librosa.feature.rms(S=S)

    sound = parselmouth.Sound(audio_path)
    harm = parselmouth.Sound.to_harmonicity_cc(sound, time_step=ts, minimum_pitch=min_pitchVal,periods_per_window=periods_per_window)
    hnr = harm.values.flatten() # Flatten hnr immediately

    # run vocal presence detector
    vp = vocal_presence(y, sr, nfft, hop_len) # vocal_presence, nfft, hop_len are global

    # Reduce features only for frames where vocal presence is detected
    f0_frames = f0[vp]
    spectral_centroid_frames = spectral_centroid.flatten()[vp]
    rms_frames = rms.flatten()[vp]

    # Align hnr with vocal presence mask and then apply mask
    if len(hnr) > len(vp): # Trim hnr if it's longer than vp
        hnr_aligned = hnr[:len(vp)]
    elif len(hnr) < len(vp): # Pad hnr if it's shorter than vp (less likely with consistent ts)
        hnr_aligned = np.pad(hnr, (0, len(vp) - len(hnr)), 'constant', constant_values=np.nan)
    else:
        hnr_aligned = hnr
    hnr_frames = hnr_aligned[vp]

    # Create a common mask for all features to ensure alignment and remove NaNs
    common_valid_mask = ~np.isnan(f0_frames) & ~np.isnan(spectral_centroid_frames) & ~np.isnan(rms_frames) & ~np.isnan(hnr_frames)

    f0_clean = f0_frames[common_valid_mask]
    rms_clean = rms_frames[common_valid_mask]
    spectral_centroid_clean = spectral_centroid_frames[common_valid_mask]
    hnr_clean = hnr_frames[common_valid_mask]

    # Calculate deltas on the cleaned arrays
    f0_delta = np.array([])
    if f0_clean.size >= 9: # Check if enough frames for default delta width=9
        try:
            f0_delta = librosa.feature.delta(f0_clean)
        except ValueError:
            f0_delta = np.array([])

    rms_delta = np.array([])
    if rms_clean.size >= 9:
        try:
            rms_delta = librosa.feature.delta(rms_clean)
        except ValueError:
            rms_delta = np.array([])

    sc_delta = np.array([])
    if spectral_centroid_clean.size >= 9:
        try:
            sc_delta = librosa.feature.delta(spectral_centroid_clean)
        except ValueError:
            sc_delta = np.array([])

    # Calculate means and percentiles
    fmin = np.min(f0_clean) if f0_clean.size > 0 else None
    fmax = np.max(f0_clean) if f0_clean.size > 0 else None
    f0_quartiles = np.percentile(f0_clean, [25, 50, 75]) if f0_clean.size > 0 else [None, None, None]

    f0_mean = np.nanmean(f0_clean) if f0_clean.size > 0 else None
    f0_delta_mean = np.nanmean(f0_delta) if f0_delta.size > 0 else None
    rms_mean = np.nanmean(rms_clean) if rms_clean.size > 0 else None
    rms_delta_mean = np.nanmean(rms_delta) if rms_delta.size > 0 else None
    spectral_centroid_mean = np.nanmean(spectral_centroid_clean) if spectral_centroid_clean.size > 0 else None
    sc_delta_mean = np.nanmean(sc_delta) if sc_delta.size > 0 else None

    # Adjusted centroid needs f0_clean and spectral_centroid_clean
    adjusted_centroid = np.divide(spectral_centroid_clean, f0_clean, where=f0_clean != 0)
    adjusted_centroid = np.nan_to_num(adjusted_centroid, nan=0.0)
    adjusted_centroid_mean = np.mean(adjusted_centroid) if adjusted_centroid.size > 0 else None

    hnr_mean = np.mean(hnr_clean) if hnr_clean.size > 0 else None

    return fmin, fmax, *f0_quartiles, f0_mean, f0_delta_mean, rms_mean, rms_delta_mean, spectral_centroid_mean, sc_delta_mean, adjusted_centroid_mean, hnr_mean

# Folder path containing audio files - Corrected to match the actual output path
folder_path = '/content/drive/'
audio_files = [f for f in os.listdir(folder_path) if f.endswith('.wav')]

# Collect results for all audio files
results = [[f, *extract_features(os.path.join(folder_path, f))] for f in audio_files] # Removed redundant arguments

# Create a DataFrame for the new features with 'name' as the first column
new_data = pd.DataFrame(results, columns=['name', 'fmin (Hz)', 'fmax (Hz)', 'F0 25th percentile (Hz)',
                                          'F0 50th percentile (Hz)', 'F0 75th percentile (Hz)', 'F0 Mean (Hz)', 'F0 Delta Mean (Hz)', 'RMS Mean', 'RMS Delta Mean', 'Spectral Centroid Mean (Hz)', 'Spectral Centroid Delta Mean (Hz)','Adjusted Centroid Mean', 'HNR Mean'])

# Save the new DataFrame to a CSV file - Corrected filename to match actual output
new_csv_path = os.path.join(folder_path, 'AudioFeatures.csv')
new_data.to_csv(new_csv_path, index=False)

print(f'New data saved to: {new_csv_path}')

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 378.8/378.8 kB 24.2 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
  Created wheel for signal_analysis: filename=Signal_Analysis-0.1.26-py3-none-any.whl size=14513 sha256=356e33a846800298bba835ab90a40a6a979c096cb589081ade1435b5081ac05e
  Stored in directory: /root/.cache/pip/wheels/8d/03/66/5b9a812973d7ab12a14b345dc87f5b56361bff3dc6d9c4d411
Successfully built signal_analysis
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.7/10.7 MB 47.4 MB/s eta 0:00:00
Mounted at /content/drive


/tmp/ipython-input-3244447384.py:176: RuntimeWarning: Mean of empty slice
  f0_mean = np.nanmean(f0_clean)
/tmp/ipython-input-3244447384.py:178: RuntimeWarning: Mean of empty slice
  rms_mean = np.nanmean(rms_clean)
/tmp/ipython-input-3244447384.py:180: RuntimeWarning: Mean of empty slice
  spectral_centroid_mean = np.nanmean(spectral_centroid_clean)
/tmp/ipython-input-3244447384.py:177: RuntimeWarning: Mean of empty slice
  f0_delta_mean = np.nanmean(f0_delta_clean) if f0_delta_clean is not None else None


New data saved to: /content/drive/MyDrive/LLMs and Gender/MusicLM_dataset/normalized_audio/AudioFeaturesDemucs.csv


In [ ]:
# get MFCC values

import os
import librosa
import numpy as np
import pandas as pd
import scipy.signal as sps


# Define paths
drive_folder_path = '/content/drive/'
existing_csv = os.path.join(drive_folder_path, 'AudioFeatures.csv')

# Load the existing CSV
if os.path.exists(existing_csv):
    existing_df = pd.read_csv(existing_csv)
else:
    raise FileNotFoundError(f"The file {existing_csv} does not exist.")

results = []

# Loop through all files in the directory
for file in os.listdir(drive_folder_path): # Use the corrected drive_folder_path
    if file.endswith(".wav"):
        file_path = os.path.join(drive_folder_path, file)

        # Load audio and extract MFCC mean values
        try:
            y, sr = librosa.load(file_path, sr=None)
        except Exception as e:
            print(f"Error loading file {file}: {e}")
            continue

        # vocal presence detector
        vp = vocal_presence(y, sr, nfft=nfft, hop_len=hop_len) # vocal_presence, nfft, hop_len are global

        mfccs = librosa.feature.mfcc(y=y, sr=sr, n_mfcc=13,n_fft=nfft, hop_length=hop_len,center=False)

        #reduce to 12 MFFCS only for frames where vocal presence is detected
        mfccs = mfccs[1:13,vp]

        mfcc_means = []
        mfcc_delta_means = []

        # Check if there are enough frames for delta calculation
        if mfccs.shape[1] >= 9:
            mfcc_means = np.mean(mfccs, axis=1).tolist()
            mfccs_delta = librosa.feature.delta(mfccs)
            mfcc_delta_means = np.mean(mfccs_delta, axis=1).tolist()
        else:

            print(f"Skipping delta calculation for {file} due to insufficient frames ({mfccs.shape[1]} frames).")
            mfcc_means = np.mean(mfccs, axis=1).tolist() if mfccs.shape[1] > 0 else [np.nan] * 12
            mfcc_delta_means = [np.nan] * 12

        # Append the results as a list
        row_data = [file]
        row_data.extend(mfcc_means)
        row_data.extend(mfcc_delta_means)
        results.append(row_data)


# DataFrame with MFCCs
mfcc_cols = [f'MFCC{i+1}' for i in range(12)]
mfcc_delta_cols = [f'MFCC Delta{i+1}' for i in range(12)]
new_df = pd.DataFrame(results, columns=['name'] + mfcc_cols + mfcc_delta_cols)

# Sort both existing_df and new_df by filename
existing_df = existing_df.sort_values(by=existing_df.columns[0]).reset_index(drop=True)
new_df = new_df.sort_values(by='name').reset_index(drop=True)

# Combine side-by-side by row order (keep all original columns)
merged = pd.concat([existing_df, new_df.drop(columns=['name'])], axis=1)

# Save back
merged.to_csv(existing_csv, index=False) # Save to the same canonical path
print(f"Updated CSV saved to: {existing_csv}")

Mounted at /content/drive
Skipping delta calculation for Woman00308.wav due to insufficient frames (0 frames).
Skipping delta calculation for Woman00380.wav due to insufficient frames (0 frames).
Skipping delta calculation for Woman00072.wav due to insufficient frames (8 frames).
Skipping delta calculation for Woman00445.wav due to insufficient frames (0 frames).
Skipping delta calculation for Woman00465.wav due to insufficient frames (0 frames).
Skipping delta calculation for Woman00481.wav due to insufficient frames (0 frames).
Skipping delta calculation for Woman00180.wav due to insufficient frames (0 frames).
Skipping delta calculation for Woman00206.wav due to insufficient frames (0 frames).
Skipping delta calculation for Woman00209.wav due to insufficient frames (6 frames).
Skipping delta calculation for Woman00561.wav due to insufficient frames (0 frames).
Skipping delta calculation for Woman00235.wav due to insufficient frames (0 frames).
Skipping delta calculation for Woman006